## actualizar retiro telef 

In [1]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text

import numpy as np

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

engine_mysql = create_engine(
    f"mysql+pymysql://{user_envio}:{pwd_envio}@{server_envio}:{port_mysql}/{db_envio}"
)


In [ ]:
lista de dni
se tomaron los separados el dia 11 y 13



In [ ]:
query = f"""
	SELECT a.*
	FROM Alice.prospectos_correos_alfin a
	INNER JOIN Alice.prospectos_envio_alfin b
		ON a.dni_cliente = b.dni_cliente
	WHERE DATE(a.fecha_envio) IN ('2026-07-16', '2026-07-17')
	AND DATE(a.fecha_envio) = DATE(b.fecha_envio);
"""
df_prospectos_correos_alfin = pd.read_sql(query, engine_mysql)

df_prospectos_correos_alfin.shape
query = f"""
	SELECT a.*
	FROM Alice.prospectos_envio_alfin a
	INNER JOIN Alice.prospectos_correos_alfin b
		ON a.dni_cliente = b.dni_cliente
	WHERE DATE(a.fecha_envio) IN ('2026-07-16', '2026-07-17')
	AND DATE(a.fecha_envio) = DATE(b.fecha_envio);
"""
df_prospectos_envio_alfin = pd.read_sql(query, engine_mysql)
df_prospectos_envio_alfin.shape

In [ ]:
query = f"""
	SELECT a.*
	FROM Alice.prospectos_correos_alfin a
	INNER JOIN Alice.prospectos_envio_alfin b
		ON a.dni_cliente = b.dni_cliente
	WHERE DATE(a.fecha_envio) IN ('2026-07-11', '2026-07-13')
	AND DATE(a.fecha_envio) = DATE(b.fecha_envio);
"""
df_prospectos_correos_alfin = pd.read_sql(query, engine_mysql)
df_prospectos_correos_alfin.sha

(1123, 20)

In [23]:
fecha_mes_base='2026-07-01'

filename='RetiroDefinitivo_BlackList.csv'
ruta_archivo = os.path.join(ruta_csv, filename)
df_def_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_BlackList.csv'
ruta_archivo = os.path.join(ruta_csv, filename)
df_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_Telefonos.csv'
ruta_archivo = os.path.join(ruta_csv, filename)
df_telf = pd.read_csv(ruta_archivo,sep='|')
filename='retiro_correo_alfin.csv'
ruta_archivo = os.path.join(ruta_csv, filename)
df_retiro_correo = pd.read_csv(ruta_archivo,sep=';')



df_def_blacklist = df_def_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_blacklist = df_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_telf= df_telf.rename(columns={'TELEFONO': 'celular'})
df_retiro_correo= df_retiro_correo.rename(columns={'DNI': 'dni_cliente'})


print(df_def_blacklist.columns.tolist())
print(df_blacklist.columns.tolist())
print(df_telf.columns.tolist())
print(df_retiro_correo.columns.tolist())

['dni_cliente']
['dni_cliente']
['celular']
['dni_cliente', 'celular', 'RETIRO']


In [24]:
# Blacklists de DNI
df1 = df_def_blacklist.copy()
df1["celular"] = None
df1 = df1[["dni_cliente", "celular"]]

df2 = df_blacklist.copy()
df2["celular"] = None
df2 = df2[["dni_cliente", "celular"]]

# Blacklist de teléfonos
df3 = df_telf.copy()
df3["dni_cliente"] = None
df3 = df3[["dni_cliente", "celular"]]

# Archivo con DNI y celular
df4 = df_retiro_correo[["dni_cliente", "celular"]].copy()

# Unir todo
df_retiros = pd.concat(
    [df1, df2, df3, df4],
    ignore_index=True
)

C:\Users\DATA\AppData\Local\Temp\ipykernel_1992\892991826.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_retiros = pd.concat(


In [25]:
df_dia=df_prospectos_correos_alfin[['dni_cliente','celular']].copy()
dni_retiro = set(df_retiros['dni_cliente'].dropna())
cel_retiro = set(df_retiros['celular'].dropna())

df_dia_retiro = df_dia[
    df_dia['dni_cliente'].isin(dni_retiro) |
    df_dia['celular'].isin(cel_retiro)
].copy()

In [21]:
df_retiro.head()

,dni_cliente,celular
0,09679313,997021829
1,09693136,936385823
2,09705504,976328585
3,09716937,918525025
4,09731063,941535397


In [12]:
df_prospectos_correos_alfin['dia'] = df_prospectos_correos_alfin['fecha_envio'].dt.day
df_prospectos_correos_alfin.groupby('dia').size().sort_index()

dia
14     744
15    2160
dtype: int64

In [29]:
df_prospectos_correos_alfin.head()

,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,color,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita,tipo_carga
0,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,09679313,MIGUEL ANGEL SEGURA SIMON,AMARILLO CLARO,1500.0,997021829,SAN JUAN DE LURIGANCHO,2026-07-18,0 days 10:30:00,MANUAL
1,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,09693136,VICTOR MANUEL YACTAYO SANCHEZ,VERDE CLARO,5200.0,936385823,VILLA EL SALVADOR 2,2026-07-18,0 days 18:15:00,MANUAL
2,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,09705504,INOCENTE YOPLA DILAS,AMARILLO OSCURO,5000.0,976328585,SAN JUAN DE MIRAFLORES,2026-07-18,0 days 09:00:00,MANUAL
3,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,09716937,JOSE LUIS MUÑANTE PEREZ,AMARILLO OSCURO,14000.0,918525025,VILLA MARIA 2,2026-07-18,0 days 12:00:00,MANUAL
4,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,09731063,BLANCA INES CHALIO BARRETO,NARANJA OSCURO,3000.0,941535397,COMAS,2026-07-18,0 days 13:30:00,MANUAL


In [ ]:
df_prospectos_correos_alfin['dia'] = df_prospectos_correos_alfin['fecha_envio'].dt.day
df_prospectos_correos_alfin.groupby('dia').size().sort_index()

In [26]:
dni_retiro = set(df_dia_retiro['dni_cliente'].dropna())

df_prospectos_correos_alfin = df_prospectos_correos_alfin[
    ~df_prospectos_correos_alfin['dni_cliente'].isin(dni_retiro)
].copy()

df_prospectos_envio_alfin = df_prospectos_envio_alfin[
    ~df_prospectos_envio_alfin['dni_cliente'].isin(dni_retiro)
].copy()

In [33]:
query = f"""
select dni_cliente from Alice.prospectos_correos_alfin
where estado='ENVIADO'
AND fecha_dia='2026-07-18'
"""
quitar = pd.read_sql(query, engine_mysql)
quitar.shape

(49, 1)

In [34]:
dni_retiro = set(quitar['dni_cliente'].dropna())

df_prospectos_correos_alfin = df_prospectos_correos_alfin[
    ~df_prospectos_correos_alfin['dni_cliente'].isin(dni_retiro)
].copy()

df_prospectos_envio_alfin = df_prospectos_envio_alfin[
    ~df_prospectos_envio_alfin['dni_cliente'].isin(dni_retiro)
].copy()

In [8]:
df_prospectos_correos_alfin.groupby('fecha_envio').size().sort_index()

fecha_envio
2026-07-11 13:25:06    1
2026-07-11 13:25:10    1
2026-07-11 13:25:13    1
2026-07-11 13:25:18    1
2026-07-11 13:25:22    1
                      ..
2026-07-11 17:45:33    1
2026-07-11 17:45:37    1
2026-07-11 17:45:42    1
2026-07-11 17:45:46    1
2026-07-11 17:45:51    1
Length: 1035, dtype: int64

In [5]:
(
    df_prospectos_correos_alfin
    .groupby('fecha_envio')
    .size()
    .reset_index(name='count')
    .sort_values('fecha_envio')
)

,fecha_envio,count
0,2026-07-11 13:25:06,1
1,2026-07-11 13:25:10,1
2,2026-07-11 13:25:13,1
3,2026-07-11 13:25:18,1
4,2026-07-11 13:25:22,1
...,...,...
1030,2026-07-11 17:45:33,1
1031,2026-07-11 17:45:37,1
1032,2026-07-11 17:45:42,1
1033,2026-07-11 17:45:46,1


In [ ]:
df_base.groupBy('fecha_envio') \
    .count() \
    .orderBy('fecha_envio') \
    .show(30)


In [13]:
df_prospectos_correos_alfin=df_prospectos_correos_alfin[['canal_campo', 'supervisor', 'ejecutivo_target', 'codigo_ejecutivo_id', 'cdv_alfin_banco', 'dni_cliente', 'nombre_cliente', 'color', 'monto_solicitado', 'celular', 'agencia_atencion', 'fecha_visita','hora_visita']]

In [14]:
df_prospectos_envio_alfin=df_prospectos_envio_alfin[['dni_vendedor', 'operador', 'dni_cliente', 'nombre_cliente', 'telefono_cliente', 'agencia_tienda', 'fecha_visita', 'monto_solicitado', 'tipo_gestion']]

In [ ]:
df_correo=df_formato[['canal_campo', 'supervisor', 'ejecutivo_target', 'codigo_ejecutivo_id', 'cdv_alfin_banco', 'dni_cliente', 'nombre_cliente', 'color', 'monto_solicitado', 'celular', 'agencia_atencion', 'fecha_visita','hora_visita']] .copy()
df_correo['tipo_carga']='MANUAL'

df_formulario=df_formato[['dni_vendedor', 'operador', 'dni_cliente', 'nombre_cliente', 'telefono_cliente', 'agencia_tienda', 'fecha_visita', 'monto_solicitado', 'tipo_gestion']].copy()

display(df_correo.head(2))
display(df_formulario.head(2))


In [45]:
df_prospectos_envio_alfin.head()

,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion
0,BOT,TARGET,09679313,MIGUEL ANGEL SEGURA SIMON,997021829,737896 - SAN JUAN DE LURIG,2026-07-18,1500,Derivacion
1,BOT,TARGET,09693136,VICTOR MANUEL YACTAYO SANCHEZ,936385823,732249 - VILLA EL SALVADOR 2,2026-07-18,5200,Derivacion
2,BOT,TARGET,09705504,INOCENTE YOPLA DILAS,976328585,738224 - SAN JUAN DE MIRAFLORES,2026-07-18,5000,Derivacion
3,BOT,TARGET,09716937,JOSE LUIS MUÑANTE PEREZ,918525025,737870 - VILLA MARIA 2,2026-07-18,14000,Derivacion
4,BOT,TARGET,09731063,BLANCA INES CHALIO BARRETO,941535397,737883 - COMAS,2026-07-18,3000,Derivacion


In [44]:
df_prospectos_correos_alfin['tipo_carga']='MANUAL'
df_prospectos_correos_alfin['fecha_visita']='2026-07-18'
df_prospectos_envio_alfin['fecha_visita']='2026-07-18'


df_prospectos_correos_alfin.to_sql(
    name="prospectos_correos_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

df_prospectos_envio_alfin.to_sql(
    name="prospectos_envio_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

C:\Users\DATA\AppData\Local\Temp\ipykernel_1992\3491498149.py:6: UserWarning: the 'timedelta' type is not supported, and will be written as integer values (ns frequency) to the database.
  df_prospectos_correos_alfin.to_sql(


2847

### seguimiento

In [ ]:
query = f"""
	SELECT a.*
	FROM Alice.prospectos_correos_alfin a
	INNER JOIN Alice.prospectos_envio_alfin b
		ON a.dni_cliente = b.dni_cliente
	WHERE DATE(a.fecha_envio) IN ('2026-07-16', '2026-07-17')
	AND DATE(a.fecha_envio) = DATE(b.fecha_envio);
"""
df_prospectos_correos_alfin_1 = pd.read_sql(query, engine_mysql)

In [37]:
dni_retiro = set(df_dia_retiro['dni_cliente'].dropna())

df_prospectos_correos_alfin_1 = df_prospectos_correos_alfin_1[
    ~df_prospectos_correos_alfin_1['dni_cliente'].isin(dni_retiro)
].copy()

dni_retiro = set(df_prospectos_correos_alfin['dni_cliente'].dropna())

df_prospectos_correos_alfin_1 = df_prospectos_correos_alfin_1[
    ~df_prospectos_correos_alfin_1['dni_cliente'].isin(dni_retiro)
].copy()


In [38]:
filename='TARGET.txt'
ruta_archivo = os.path.join(ruta_csv, filename)
df_desembolso = pd.read_csv(ruta_archivo,sep='|')
df_desembolso.head()

,DNI,CUENTA_BT,N_OPER,COD_SUCURSAL,SUCURSAL,MONTO_FINANCIADO,FECHA_SOL,FECHA_DESEMBOLSOS,TEA,CANAL,TIPO_DESEM,CODIGO_ID
0,27386106,18198283,8465484,4281,CHICLAYO BALTA,9200.0,2026-07-17,2026-07-17,65.5,TARGET,DERIVACION,1
1,32760225,18198239,8465400,8381,EMANCIPACION,3000.0,2026-07-17,2026-07-17,75.0,TARGET,DERIVACION,00000001
2,40507683,18198213,8465409,4270,SULLANA,6000.0,2026-07-17,2026-07-17,78.0,TARGET,DERIVACION,00000001
3,40846124,18198312,8465536,2249,VILLA EL SALVADOR 2,16300.0,2026-07-17,2026-07-17,53.0,TARGET,DERIVACION,00000001
4,44570838,5018247,8465508,4270,SULLANA,2000.0,2026-07-17,2026-07-17,102.0,TARGET,DERIVACION,00000001


In [39]:
dni_retiro = set(df_desembolso['DNI'].dropna())

df_prospectos_correos_alfin_1 = df_prospectos_correos_alfin_1[
    ~df_prospectos_correos_alfin_1['dni_cliente'].isin(dni_retiro)
].copy()


In [40]:
ruta_archivo = os.path.join(ruta_csv, 'seguimiento_alfin.xlsx')
df_prospectos_correos_alfin_1.to_excel(ruta_archivo, index=False)

In [42]:
df_prospectos_correos_alfin_1.shape

(3039, 20)

In [11]:
df_prospectos_envio_alfin.head()

,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion
0,BOT,TARGET,15343243,JOSE LUIS SANCHEZ VELASQUEZ,940458253,732243 - CAÑETE,2026-07-09,14000,Derivacion
1,BOT,TARGET,23945463,NICOLAS HUGO CABANA HUAMANI,969259975,734299 - CUSCO LA CULTURA,2026-07-09,2000,Derivacion
2,BOT,TARGET,41807964,ELIZABETH ALBINA CAJALEON FLORES,991077480,734280 - PC HUANCAYO,2026-07-09,7100,Derivacion
3,BOT,TARGET,73947051,LIDALINA AUQUI ÑACAYAURI,929350308,732249 - VILLA EL SALVADOR 2,2026-07-09,3400,Derivacion
4,BOT,TARGET,22966170,JUAN CARLOS LOPEZ OMONTE,944481790,735996 - HUANUCO,2026-07-09,1600,Derivacion


In [30]:
df_prospectos_correos_alfin.shape

(2896, 14)

agregando retiro

In [27]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text

fecha_mes_base='2026-07-01'

filename='RetiroDefinitivo_BlackList.csv'
ruta_archivo = os.path.join(ruta_csv, filename)
df_def_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_BlackList.csv'
ruta_archivo = os.path.join(ruta_csv, filename)
df_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_Telefonos.csv'
ruta_archivo = os.path.join(ruta_csv, filename)
df_telf = pd.read_csv(ruta_archivo,sep='|')

filename='retiro_correo_alfin.csv'
ruta_archivo = os.path.join(ruta_csv, filename)
df_retiro_correo = pd.read_csv(ruta_archivo,sep=';')

df_def_blacklist = df_def_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_blacklist = df_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_telf= df_telf.rename(columns={'TELEFONO': 'celular'})
df_retiro_correo= df_retiro_correo.rename(columns={'DNI': 'dni_cliente'})

print(df_def_blacklist.columns)
print(df_blacklist.columns)
print(df_telf.columns)
print(df_retiro_correo.columns)
print(df_formato.columns.tolist())

Index(['dni_cliente'], dtype='object')
Index(['dni_cliente'], dtype='object')
Index(['celular'], dtype='object')
Index(['dni_cliente', 'celular', 'RETIRO'], dtype='object')
['dni_cliente', 'nombre_cliente', 'celular', 'cod_agencia', 'agencia_atencion', 'fecha_visita', 'monto_solicitado', 'color', 'supervisor', 'canal_campo', 'codigo_ejecutivo_id', 'ejecutivo_target', 'cdv_alfin_banco', 'hora_visita', 'telefono_cliente', 'dni_vendedor', 'agencia_tienda', 'operador', 'tipo_gestion']


In [28]:
df_def_blacklist["dni_cliente"] = (
    df_def_blacklist["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)
df_blacklist["dni_cliente"] = (
    df_blacklist["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)
df_retiro_correo["dni_cliente"] = (
    df_retiro_correo["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)


In [29]:
dni_blacklist = set(df_def_blacklist["dni_cliente"]).union(
    set(df_blacklist["dni_cliente"])
).union(
    set(df_retiro_correo["dni_cliente"])
)


df_formato = df_formato[
    ~df_formato["dni_cliente"].isin(dni_blacklist)
]

telefonos = set(df_telf["celular"]).union(
    set(df_retiro_correo["celular"])
)

df_formato = df_formato[
    ~df_formato["celular"].isin(telefonos)
]

In [ ]:

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

server_sql = server_zeus
db_sql = "SAMANTHA"
user_sql = user_zeus
pwd_sql = pwd_zeus

engine_samantha = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

In [13]:
df_formato.drop_duplicates(subset=["dni_cliente"], inplace=True)



In [7]:
df_formato.shape

(2191, 20)

In [20]:
# df_formato = df_formato[
#     df_formato['agencia_atencion'].isin([
#         'SAN JUAN DE LURIG',
#         'ENMANCIPACION',
#         'PC HUANCAYO',
#         'TRUJ CENTRO',
#         'PC TACNA',
#         'PC HUARAZ',
#         'TRUJ AMERICA',
#         'AREQ CAYMA',
#         'AREQ PAMPILLA'
#     ])
# ]

#### Validar el nombre de la agencia

In [32]:
query = f"""
	select * from Alice.agencias_alfin
"""
df_agencia = pd.read_sql(query, engine_mysql)

set_correo = set(
    df_formato['agencia_atencion']
    .dropna()
    .drop_duplicates()
)

set_agencia = set(
    df_agencia['agencia_correo']
    .dropna()
    .drop_duplicates()
)
# print(set_correo & set_agencia)
print(set_agencia - set_correo)
print(set_correo -set_agencia )

{'IQUITOS'}
set()


In [15]:
print(set_agencia - set_correo)
print(set_correo -set_agencia )

{'IQUITOS'}
{'TE'}


#### validar el codigo de agencia 

In [31]:
equivalencias = {
    'SAN JUAN DE LURIG': 'SAN JUAN DE LURIGANCHO',
    'SAN JUAN DE LURIG': 'SAN JUAN DE LURIGANCHO',
    'ENMANCIPACION': 'EMANCIPACION',
    'PC HUANCAYO': 'HUANCAYO',
    'PC TACNA': 'TACNA',
    'PC HUARAZ': 'HUARAZ',
    'TRUJ CENTRO': 'TRUJILLO CENTRO',
    'TRUJ AMERICA': 'TRUJILLO AMERICA',
    'AREQ CAYMA': 'AREQUIPA CAYMA',
    'AREQ PAMPILLA': 'AREQUIPA PAMPILLA'
}

df_formato['agencia_atencion'] = (
    df_formato['agencia_atencion']
    .replace(equivalencias)
)

In [32]:
df_agencia[df_agencia['agencia_correo']=='SAN JUAN DE MIRAFLORES'].head()


,agencia_Formulario,agencia_base,agencia_base2,agencia_correo,correos
34,738224 - SAN JUAN DE MIRAFLORES,SAN JUAN DE MIRAFLORES,SAN JUAN DE MIRAFLORES,SAN JUAN DE MIRAFLORES,yolinda.huayca@alfinbanco.pe


In [30]:
df_correo[df_correo['agencia_atencion']=='SAN JUAN DE MIRAFLORES'].head()


,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,color,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita,tipo_carga
74,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,01236008,HERMENEGILDO CALSIN YUCRA,NaN,10000,952524186,SAN JUAN DE MIRAFLORES,2026-07-15,18:30:00,MANUAL
93,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,15627921,MARIZA OBDULIA ROSALES GIRIO DE CHINCHAY,VERDE OSCURO,9700,924119267,SAN JUAN DE MIRAFLORES,2026-07-15,12:00:00,MANUAL
132,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,16806886,VIDELMO YLATOMA BUSTAMANTE,NaN,8900,949886328,SAN JUAN DE MIRAFLORES,2026-07-15,14:45:00,MANUAL
192,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,00205546,ARTURO CRUZ CAMPAÑA,NaN,6000,962095319,SAN JUAN DE MIRAFLORES,2026-07-15,14:30:00,MANUAL
216,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,00214859,ENA ROSA BARRETO DIOSES,NaN,5200,922172667,SAN JUAN DE MIRAFLORES,2026-07-15,09:30:00,MANUAL


In [33]:

set_correo = set(
    df_formato['agencia_tienda']
    .dropna()
    .drop_duplicates()
)

set_agencia = set(
    df_agencia['agencia_Formulario']
    .dropna()
    .drop_duplicates()
)
# print(set_correo & set_agencia)
print(set_agencia - set_correo)
print(set_correo -set_agencia )

set()
{'VILLA MARIA 2', 'PC HUANCAYO', 'AREQUIPA PAMPILLA', 'CHIMBOTE', 'SAN JUAN DE MIRAFLORES'}


In [19]:
print(df_agencia["agencia_correo"].drop_duplicates().tolist())

['CAJAMARCA', 'CASTILLA', 'CHICLAYO BALTA', 'CHIMBOTE', 'MOSHOQUEQUE', None, 'HUARAZ', 'SULLANA', 'TRUJILLO AMERICA', 'TRUJILLO CENTRO', 'AREQUIPA CAYMA', 'AREQUIPA PAMPILLA', 'CAÑETE', 'CHINCHA', 'CUSCO LA CULTURA', 'HUACHO', 'HUANUCO', 'HUARAL', 'ICA', 'IQUITOS', 'JULIACA 2', 'HUANCAYO', 'TACNA', 'PISCO', 'PUCALLPA', 'TARAPOTO', 'ATE VITARTE', 'COMAS', 'EMANCIPACION', 'JESUS MARIA', 'LOS OLIVOS', 'MIRAFLORES', 'PUENTE PIEDRA', 'SAN JUAN DE LURIGANCHO', 'SAN JUAN DE MIRAFLORES', 'SAN MARTIN', 'SAN MIGUEL', 'SANTA ANITA', 'VENTANILLA', 'VILLA EL SALVADOR 2', 'VILLA MARIA 2', 'TUMBES']


In [ ]:
['CAJAMARCA', 'CASTILLA', 'CHICLAYO BALTA', 'CHIMBOTE', 'MOSHOQUEQUE', None, 'HUARAZ', 'SULLANA', 'TRUJILLO AMERICA', 'TRUJILLO CENTRO', 'AREQUIPA CAYMA', 'AREQUIPA PAMPILLA', 'CAÑETE', 'CHINCHA', 'CUSCO LA CULTURA', 'HUACHO', 'HUANUCO', 'HUARAL', 'ICA', 'IQUITOS', 'JULIACA 2', 'HUANCAYO', 'TACNA', 'PISCO', 'PUCALLPA', 'TARAPOTO', 'ATE VITARTE', 'COMAS', 'EMANCIPACION', 'JESUS MARIA', 'LOS OLIVOS', 'MIRAFLORES', 'PUENTE PIEDRA', 'SAN JUAN DE LURIGANCHO', 'SAN JUAN DE MIRAFLORES', 'SAN MARTIN', 'SAN MIGUEL', 'SANTA ANITA', 'VENTANILLA', 'VILLA EL SALVADOR 2', 'VILLA MARIA 2', 'TUMBES']MARIA 

In [ ]:
    {'VILLA MARIA 2', 'PC HUANCAYO', 'AREQUIPA PAMPILLA', 'CHIMBOTE', 'SAN JUAN DE MIRAFLORES'}

equivalencias = {
    'SAN JUAN DE LURIG': 'SAN JUAN DE LURIGANCHO',
    'ENMANCIPACION': 'EMANCIPACION',
    'PC HUANCAYO': 'HUANCAYO',
    'PC TACNA': 'TACNA',
    'PC HUARAZ': 'HUARAZ',
    'TRUJ CENTRO': 'TRUJILLO CENTRO',
    'TRUJ AMERICA': 'TRUJILLO AMERICA',
    'AREQ CAYMA': 'AREQUIPA CAYMA',
    'AREQ PAMPILLA': 'AREQUIPA PAMPILLA'
}

df_formato['agencia_atencion'] = (
    df_formato['agencia_atencion']
    .replace(equivalencias)
)
# df_formato.drop_duplicates(subset=["dni"], inplace=True)

In [34]:
print(df_correo["agencia_atencion"].drop_duplicates().tolist())


['HUANUCO', 'HUANCAYO', 'SAN MIGUEL', 'CUSCO LA CULTURA', 'TACNA', 'TRUJILLO CENTRO', 'TARAPOTO', 'EMANCIPACION', 'CHIMBOTE', 'MIRAFLORES', 'TRUJILLO AMERICA', 'HUACHO', 'COMAS', 'CHICLAYO BALTA', 'SAN JUAN DE LURIGANCHO', 'CASTILLA', 'AREQUIPA PAMPILLA', 'SULLANA', 'VENTANILLA', 'MOSHOQUEQUE', 'JESUS MARIA', 'PUCALLPA', 'ATE VITARTE', 'LOS OLIVOS', 'VILLA MARIA 2', 'SANTA ANITA', 'CAJAMARCA', 'AREQUIPA CAYMA', 'PISCO', 'HUARAZ', 'SAN JUAN DE MIRAFLORES', 'CHINCHA', 'HUARAL', 'ICA', 'SAN MARTIN', 'JULIACA 2', 'VILLA EL SALVADOR 2', 'TUMBES', 'CAÑETE', 'PUENTE PIEDRA', nan, 'TE']


In [ ]:

df_correo[df_correo['dni_cliente']=='09704310'].head()

,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,color,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita,tipo_carga
2190,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,09704310,SALVADOR ALBERTO CHOQUE ALARCON,NaN,18000,930162239,MIRAFLORES,2026-07-15,13:30:00,MANUAL


In [ ]:
df_correo=df_formato[['canal_campo', 'supervisor', 'ejecutivo_target', 'codigo_ejecutivo_id', 'cdv_alfin_banco', 'dni_cliente', 'nombre_cliente', 'color', 'monto_solicitado', 'celular', 'agencia_atencion', 'fecha_visita','hora_visita']] .copy()
df_correo['tipo_carga']='MANUAL'

df_formulario=df_formato[['dni_vendedor', 'operador', 'dni_cliente', 'nombre_cliente', 'telefono_cliente', 'agencia_tienda', 'fecha_visita', 'monto_solicitado', 'tipo_gestion']].copy()

display(df_correo.head(2))
display(df_formulario.head(2))


,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,color,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita,tipo_carga
0,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,0000None,None,None,None,None,None,2026-07-16,15:15:00,MANUAL
1,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,00002596,LILIA CACHIQUE LOPEZ,NARANJA OSCURO,3000,978024945,PUCALLPA,2026-07-16,18:15:00,MANUAL


,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion
0,BOT,TARGET,0000None,None,None,None,2026-07-16,None,Derivacion
1,BOT,TARGET,00002596,LILIA CACHIQUE LOPEZ,978024945,738334 - PUCALLPA,2026-07-16,3000,Derivacion


In [34]:
df_correo['fecha_visita']='2026-07-16'
df_formulario['fecha_visita']='2026-07-16'


In [17]:
df_prospectos_correos_alfin[df_prospectos_correos_alfin['dni_cliente']=='09704310'].head()


,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,color,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita,tipo_carga
2903,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,09704310,SALVADOR ALBERTO CHOQUE ALARCON,None,18000.0,930162239,MIRAFLORES,2026-07-18,0 days 13:30:00,MANUAL


In [16]:


df_prospectos_correos_alfin.to_sql(
    name="prospectos_correos_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

df_prospectos_envio_alfin.to_sql(
    name="prospectos_envio_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

C:\Users\DATA\AppData\Local\Temp\ipykernel_1992\3762344665.py:1: UserWarning: the 'timedelta' type is not supported, and will be written as integer values (ns frequency) to the database.
  df_prospectos_correos_alfin.to_sql(


2904

In [ ]:
from sqlalchemy import text

query = """
UPDATE crm_target.alfin_clientes a
INNER JOIN crm_target.tb_temporal b
    ON a.NUMERO_DOCUMENTO = b.col_01 
SET 
    a.estado = b.col_02,
    a.cl_estado= b.col_03
WHERE 
    a.cl_base = 'mayo 2026'
"""

with engine_mysql.begin() as conn:
    result = conn.execute(text(query))
    print("Filas afectadas:", result.rowcount)

In [35]:
df_correo.shape

(5060, 14)